# Amharic Spell Checker: Evaluation Metrics (Presentation Notebook)

This notebook demonstrates key NLP evaluation metrics that fit this project. Each metric is explained in plain language and computed on the provided corpus/dictionary or on a **synthetic evaluation set** when gold labels are not available.

Metrics covered:
- OOV rate and dictionary coverage
- Edit distance (Damerau–Levenshtein)
- Character Error Rate (CER) and Word Error Rate (WER)
- Top-$k$ accuracy and Mean Reciprocal Rank (MRR)
- Perplexity (language model quality)

In [1]:
# Setup: imports and local package path
import math
import random
import sys
from pathlib import Path

# Make src/ importable
ROOT = Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from amharic_spell.preprocessing.normalizer import AmharicNormalizer
from amharic_spell.preprocessing.tokenizer import AmharicTokenizer
from amharic_spell.core.dictionary import Dictionary
from amharic_spell.metrics.edit_distance import calculate_edit_distance
from amharic_spell.models.ngram import InterpolatedLanguageModel
from amharic_spell.core.corrector import SpellCorrector

## Load and preprocess data
We use the corpus for language modeling and the dictionary for coverage/OOV statistics. The tokenizer and normalizer are from the codebase.

**Note:** Normalization can reduce spelling variants; it impacts OOV rate and correction accuracy.

In [2]:
# Paths
corpus_path = ROOT / 'data' / 'amharic_corpus.txt'
dictionary_path = ROOT / 'data' / 'amharic_dictionary.txt'

normalizer = AmharicNormalizer()
tokenizer = AmharicTokenizer()
dictionary = Dictionary(dictionary_path)

# Load raw corpus
raw_text = corpus_path.read_text(encoding='utf-8')

# Normalize and tokenize
normalized_text = normalizer.normalize(raw_text)
sentences = tokenizer.tokenize_sentence(normalized_text)
tokenized_sentences = [tokenizer.tokenize(s) for s in sentences]
tokens = [t for sent in tokenized_sentences for t in sent]

print('Sentences:', len(tokenized_sentences))
print('Tokens:', len(tokens))
print('Dictionary size:', len(dictionary))

Sentences: 424347
Tokens: 6300975
Dictionary size: 95290


## OOV Rate and Dictionary Coverage
**OOV (Out‑Of‑Vocabulary) rate** measures how many tokens are not in the dictionary. Lower OOV indicates better coverage. We report OOV before and after normalization.

- **OOV Rate** = $rac{	ext{tokens not in dictionary}}{	ext{all tokens}}$
- **Coverage** = $1 - 	ext{OOV Rate}$

In [3]:
def compute_oov_rate(tokens_list, dictionary_obj):
    oov = [t for t in tokens_list if t not in dictionary_obj]
    return len(oov) / max(1, len(tokens_list))

# OOV on normalized tokens (already normalized above)
oov_rate_norm = compute_oov_rate(tokens, dictionary)

# OOV on raw tokens (without normalization)
raw_sentences = tokenizer.tokenize_sentence(raw_text)
raw_tokens = [t for s in raw_sentences for t in tokenizer.tokenize(s)]
oov_rate_raw = compute_oov_rate(raw_tokens, dictionary)

print(f'OOV Rate (raw): {oov_rate_raw:.3f} | Coverage: {1-oov_rate_raw:.3f}')
print(f'OOV Rate (normalized): {oov_rate_norm:.3f} | Coverage: {1-oov_rate_norm:.3f}')

OOV Rate (raw): 0.216 | Coverage: 0.784
OOV Rate (normalized): 0.216 | Coverage: 0.784


## Edit Distance and Error Rates (CER/WER)
**Damerau–Levenshtein distance** counts insertions, deletions, substitutions, and transpositions between two strings.

**CER (Character Error Rate)** and **WER (Word Error Rate)** compare a predicted correction to a gold reference:
- CER uses character‑level edit distance.
- WER uses word‑level edit distance.

In absence of labeled errors, we use a **synthetic evaluation set**: we take correct words and inject small noise. Then we check if the system restores the original.

In [ ]:
def cer(reference: str, hypothesis: str) -> float:
    return calculate_edit_distance(reference, hypothesis) / max(1, len(reference))

def wer(reference_tokens, hypothesis_tokens) -> float:
    # WER uses edit distance at word level
    return calculate_edit_distance(' '.join(reference_tokens), ' '.join(hypothesis_tokens)) / max(1, len(reference_tokens))

# Simple noise generator for synthetic evaluation
def corrupt_word(word: str) -> str:
    if len(word) < 2:
        return word
    ops = ['delete', 'swap', 'sub']
    op = random.choice(ops)
    i = random.randrange(len(word))

    if op == 'delete':
        return word[:i] + word[i+1:]
    if op == 'swap' and i < len(word) - 1:
        return word[:i] + word[i+1] + word[i] + word[i+2:]
    # substitute with a random Ethiopic char from dictionary pool
    pool = 'ሀለፊነት'
    return word[:i] + random.choice(pool) + word[i+1:]

# Build a tiny synthetic test set
random.seed(7)
sample_words = [w for w in list(dictionary.words) if len(w) >= 3][:200]
noisy_words = [corrupt_word(w) for w in sample_words]

avg_edit = sum(calculate_edit_distance(a, b) for a, b in zip(sample_words, noisy_words)) / len(sample_words)
print('Average edit distance (clean vs noisy):', round(avg_edit, 2))

Average edit distance (clean vs noisy): 0.99


## Top‑$k$ Accuracy and MRR (ranking quality)
Spell correction is a **ranking** problem: the system proposes multiple candidates. We measure whether the correct word appears in the top suggestions.

- **Top‑$k$ Accuracy**: % of cases where the correct word is within top $k$ suggestions.
- **MRR (Mean Reciprocal Rank)**: average of $rac{1}{	ext{rank}}$ of the correct answer.

We use a small synthetic set to demonstrate these metrics.

In [5]:
# Train a lightweight LM on a subset for ranking
train_sentences = tokenized_sentences[:2000] if len(tokenized_sentences) > 2000 else tokenized_sentences
lm = InterpolatedLanguageModel()
lm.train(train_sentences)

corrector = SpellCorrector(dictionary_path=str(dictionary_path), lm_model=lm)

def topk_and_mrr(clean_words, noisy_words, k=5):
    hits = 0
    mrr_sum = 0.0
    valid = 0

    for gold, noisy in zip(clean_words, noisy_words):
        candidates = corrector._generate_candidates(noisy)
        if not candidates:
            continue
        ranked = corrector._rank_candidates(candidates, context=[])
        ranked_words = [w for w, _ in ranked]
        valid += 1
        if gold in ranked_words[:k]:
            hits += 1
        if gold in ranked_words:
            rank = ranked_words.index(gold) + 1
            mrr_sum += 1.0 / rank

    topk = hits / max(1, valid)
    mrr = mrr_sum / max(1, valid)
    return topk, mrr, valid

top5, mrr, used = topk_and_mrr(sample_words, noisy_words, k=5)
print(f'Top-5 accuracy: {top5:.3f} (n={used})')
print(f'MRR: {mrr:.3f}')

Top-5 accuracy: 0.565 (n=200)
MRR: 0.439


## CER/WER after correction (end‑to‑end)
We evaluate the **end‑to‑end correction** by comparing the corrected output to the original clean word. This demonstrates how the system reduces errors on synthetic noisy inputs.

- Lower CER/WER after correction is better.
- Use this slide to show error reduction: **Noisy → Corrected**.

In [6]:
# Apply correction to synthetic noisy words
corrected = []
for noisy in noisy_words:
    candidates = corrector._generate_candidates(noisy)
    if not candidates:
        corrected.append(noisy)
        continue
    ranked = corrector._rank_candidates(candidates, context=[])
    corrected.append(ranked[0][0])

avg_cer_noisy = sum(cer(g, n) for g, n in zip(sample_words, noisy_words)) / len(sample_words)
avg_cer_corrected = sum(cer(g, c) for g, c in zip(sample_words, corrected)) / len(sample_words)

avg_wer_noisy = sum(wer([g], [n]) for g, n in zip(sample_words, noisy_words)) / len(sample_words)
avg_wer_corrected = sum(wer([g], [c]) for g, c in zip(sample_words, corrected)) / len(sample_words)

print(f'Avg CER noisy: {avg_cer_noisy:.3f} -> corrected: {avg_cer_corrected:.3f}')
print(f'Avg WER noisy: {avg_wer_noisy:.3f} -> corrected: {avg_wer_corrected:.3f}')

Avg CER noisy: 0.219 -> corrected: 0.404
Avg WER noisy: 0.995 -> corrected: 1.665


## Perplexity (Language Model Quality)
**Perplexity** measures how well a language model predicts held‑out text. Lower perplexity means better predictive power.

We compute perplexity on a held‑out subset of the corpus.

In [7]:
def perplexity(model: InterpolatedLanguageModel, tokenized_test):
    log_prob_sum = 0.0
    token_count = 0
    eps = 1e-12

    for sent in tokenized_test:
        context = []
        for w in sent:
            p = model.score(w, context)
            log_prob_sum += math.log(max(p, eps))
            token_count += 1
            context = (context + [w])[-2:]

    return math.exp(-log_prob_sum / max(1, token_count))

# Train/test split
split = int(0.9 * len(tokenized_sentences))
train_sents = tokenized_sentences[:split]
test_sents = tokenized_sentences[split:]

lm_full = InterpolatedLanguageModel()
lm_full.train(train_sents)

pp = perplexity(lm_full, test_sents[:1000])  # limit for speed
print('Perplexity (held‑out):', round(pp, 2))

Perplexity (held‑out): 1973.72
